# Task 2: MuRIL fine-tuning
Fine-tunes [`google/muril-base-cased`](https://huggingface.co/google/muril-base-cased) with a 3-way classification head on the **raw** (non-normalized) training tweets. The test split is then scored twice: on raw text, and on text normalized with Task 1's Dakshina normalization, without retraining.

- **Inputs:** `data/partition_b_*_processed.json` (from Task 1)
- **Outputs:** `results/task2_metrics.json`, `results/task2_predictions.json`
- **Runtime:** **T4 GPU** (Runtime → Change runtime type), about 9 minutes.

## Setup

In [ ]:
# Work in a folder on Google Drive so all notebooks share data/ and results/
import os
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/NLP_Group17"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))

## Configuration

In [ ]:
import os
import json
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, classification_report
from tqdm import tqdm

SEED = 42
MODEL_NAME = "google/muril-base-cased"
DATA_DIR = "data"
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## Dataset
Tokenizes each tweet to at most 128 subwords. `text_key` selects the raw or the normalized text.

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, items, tokenizer, text_key="raw_text", max_len=128):
        self.items = items
        self.tokenizer = tokenizer
        self.text_key = text_key
        self.max_len = max_len

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        text = item[self.text_key]
        label = LABEL2ID[item["sentiment"]]
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "id": item["id"],
            "text": text,
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

## Evaluation
Returns macro-F1, accuracy and per-tweet predictions.

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, records = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
            for tid, txt, y_true, y_pred in zip(batch["id"], batch["text"], labels.cpu().numpy(), preds.cpu().numpy()):
                records.append({
                    "id": tid,
                    "text": txt,
                    "gold": ID2LABEL[int(y_true)],
                    "pred": ID2LABEL[int(y_pred)]
                })
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    acc = accuracy_score(all_labels, all_preds)
    return macro_f1, acc, all_labels, all_preds, records

## Training and testing
AdamW (learning rate 2e-5, weight decay 0.01), batch size 16, 4 epochs, linear warmup over the first 10% of steps, gradient clipping at 1.0. The epoch with the best validation macro-F1 is evaluated on the raw and on the normalized test text.

In [ ]:
def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    with open(os.path.join(DATA_DIR, "partition_b_train_processed.json"), "r", encoding="utf-8") as f:
        train_data = json.load(f)
    with open(os.path.join(DATA_DIR, "partition_b_validation_processed.json"), "r", encoding="utf-8") as f:
        val_data = json.load(f)
    with open(os.path.join(DATA_DIR, "partition_b_test_processed.json"), "r", encoding="utf-8") as f:
        test_data = json.load(f)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)

    train_loader = DataLoader(TweetDataset(train_data, tokenizer, "raw_text", 128), batch_size=16, shuffle=True)
    val_loader = DataLoader(TweetDataset(val_data, tokenizer, "raw_text", 128), batch_size=32, shuffle=False)
    test_raw_loader = DataLoader(TweetDataset(test_data, tokenizer, "raw_text", 128), batch_size=32, shuffle=False)
    test_norm_loader = DataLoader(TweetDataset(test_data, tokenizer, "norm_text", 128), batch_size=32, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    epochs = 4
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

    best_val_f1 = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Task 2 Epoch {epoch}/{epochs}"):
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        val_f1, val_acc, _, _, _ = evaluate(model, val_loader, device)
        print(f"Epoch {epoch} | Train Loss: {total_loss/len(train_loader):.4f} | Val Macro-F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    # 1. Standard Test Evaluation (Raw Input)
    raw_f1, raw_acc, y_true, y_pred_raw, raw_records = evaluate(model, test_raw_loader, device)
    print(f"\nTask 2 Test Macro-F1 (Raw Input):        {raw_f1:.4f} (Acc: {raw_acc:.4f})")
    print(classification_report(y_true, y_pred_raw, target_names=["negative", "neutral", "positive"], digits=4))

    # 2. Additional Check: Test-Time Dakshina Normalization Only (No Retraining)
    norm_f1, norm_acc, _, y_pred_norm, _ = evaluate(model, test_norm_loader, device)
    print(f"Task 2 Test Macro-F1 (Normalized Test):  {norm_f1:.4f} (Acc: {norm_acc:.4f})")

    results = {
        "task2_val_macro_f1": float(best_val_f1),
        "task2_test_raw_macro_f1": float(raw_f1),
        "task2_test_raw_accuracy": float(raw_acc),
        "task2_test_norm_only_macro_f1": float(norm_f1),
        "task2_test_norm_only_accuracy": float(norm_acc)
    }
    with open(os.path.join(RESULTS_DIR, "task2_metrics.json"), "w") as f:
        json.dump(results, f, indent=2)
    with open(os.path.join(RESULTS_DIR, "task2_predictions.json"), "w", encoding="utf-8") as f:
        json.dump(raw_records, f, ensure_ascii=False, indent=2)

## Run

In [ ]:
main()

Metrics are in `results/task2_metrics.json`. Next: `task3/task3_lid_muril.ipynb`.